In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import warnings
from pathlib import Path

model = "MIROC6"
exp = "historical"

# TODO:better file loading
mask_file = f"/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/valid_event_days/{model}_{exp}_valid_event_days_1850-2014.nc"
et_90_file = f"/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/90th_percentiles/{model}_historical_90th_percentiles.nc"
data_file = f"/work10/archive/CMIP6/CMIP-SSPs/outputs/{model}/{exp}/{model}_{exp}_daily_ET0_18500101-20141231.nc"

mask_ds = xr.open_dataset(mask_file)
p90_ds = xr.open_dataset(et_90_file)
data_ds = xr.open_dataset(data_file)

In [2]:
# Helper functions
def get_growing_month_mask(lat, lon, month):
    """
    Growing season definition:
    NH: Apr-Oct
    SH: Oct-Apr
    """
    out = np.zeros(shape=(len(lat), len(lon)), dtype=bool)

    nh = lat >= 0
    sh = lat < 0

    # GS in NH is from April to October
    out[nh] = (4 <= month <= 10)
    # GS in SH is October to December and January to April
    out[sh] = (month >= 10) | (month <= 4)

    return out # array with shape len(lat), len(lon).

In [ ]:
et0_rad_file = f"/work10/archive/CMIP6/CMIP-SSPs/outputs/{model}/{exp}/{model}_{exp}_daily_ET0rad_18500101-20141231.nc"
et0_adv_file = f"/work10/archive/CMIP6/CMIP-SSPs/outputs/{model}/{exp}/{model}_{exp}_daily_ET0adv_18500101-20141231.nc"

CHUNKS = {'time': 365}
rad = xr.open_dataset(et0_rad_file, chunks=CHUNKS)
adv = xr.open_dataset(et0_adv_file, chunks=CHUNKS)

# Drop Feb. 29th
rad = rad.convert_calendar('noleap', align_on='year')["ET0rad"]
adv = adv.convert_calendar('noleap', align_on='year')["ET0adv"]

# Daily climatologies from 1981–2000 baseline
rad_cli = rad.sel(time=slice('1981', '2000')).groupby('time.dayofyear').mean()
adv_cli = adv.sel(time=slice('1981', '2000')).groupby('time.dayofyear').mean()

# Compute rad_frac (fractional contribution of radiative term)
rad_anom = rad.groupby(rad.time.dt.dayofyear) - rad_cli
adv_anom = adv.groupby(adv.time.dt.dayofyear) - adv_cli
denom = (rad_anom + adv_anom)
rad_frac = rad_anom / denom

# If denominator is negative or rad_frac is infinite, fill grid cell with NaN
rad_frac = xr.where(cond=((denom <= 0) | (~np.isfinite(rad_frac))),
                      x=np.nan, y=rad_frac)

rad_frac_ds = rad_frac.to_dataset(name="rad_frac")

filename = f"/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/rad_frac/{model}/{model}_{exp}_daily_rad_frac_18500101-20141231.nc"
rad_frac_ds.to_netcdf(filename)

outdir = Path(".")
outdir.mkdir(exist_ok=True)
outpath = outdir / f"annual_thirstwave_metrics_v2.png"
fig.savefig(outpath, dpi=300, bbox_inches="tight")
print(f"\n\tSaved figure to {outpath.resolve()}")

In [ ]:
rad_anom_sample = rad_anom.sel(time='2005-11-08').sel(lat=slice(0, 10), lon=slice(0,10)).values
rad_anom_sample = rad_anom_sample.reshape(7, 8)

rad_sample = rad.sel(time='2005-11-08').sel(lat=slice(0, 10), lon=slice(0,10)).values
rad_sample = rad_sample.reshape(7, 8)

rad_cli_sample = rad_cli.sel(dayofyear=312).sel(lat=slice(0, 10), lon=slice(0,10)).values
rad_cli_sample = rad_cli_sample.reshape(7, 8)

import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))
kwargs = {'annot': True, 'vmin': -5, 'vmax': 5, 'cmap': 'PuOr'}
sns.heatmap(rad_anom_sample, ax=axes[0], **kwargs)
sns.heatmap(rad_sample, ax=axes[1], **kwargs)
sns.heatmap(rad_cli_sample, ax=axes[2], **kwargs)

axes[0].set_title("Radiative term anomaly")
axes[1].set_title("Radiative term")
axes[2].set_title("Daily climatology")
fig.suptitle("Radiative term anomaly on Nov. 8th, 2005 (selected grid cells)")

In [4]:
def calculate_event_based_metrics(data_ds: xr.Dataset, mask_ds: xr.Dataset, p90_ds: xr.Dataset, rad_frac_ds: xr.Dataset) -> xr.Dataset:
    mask_da = mask_ds["ET0"]
    data_da = data_ds["ET0"]
    rad_frac_da = rad_frac_ds["rad_frac"]
    p90_da = p90_ds["ET0"]

    # Sanity check: all three must sit on the same grid
    if not all(np.equal(data_da.lat.values, mask_da.lat.values)):
        raise ValueError("mask and data have different latitude values")
    if not all(np.equal(data_da.lon.values, mask_da.lon.values)):
        raise ValueError("mask and data have different longitude values")
    if not all(np.equal(p90_da.lat.values, mask_da.lat.values)):
        raise ValueError("p90 climatology has different latitude values than mask/data")
    if not all(np.equal(p90_da.lon.values, mask_da.lon.values)):
        raise ValueError("p90 climatology has different longitude values than mask/data")

    # The detection algorithm performs bitwise AND and OR operations, which may fail if mask values are integers
    if mask_da.dtype != bool:
        raise ValueError(f"The valid event days mask should contain boolean values but type is {mask_da.dtype}")

    p90_dim0 = p90_da.dims[0]
    if p90_da.sizes[p90_dim0] != 365:
        raise ValueError(f"expected a 365-day p90 climatology, got {p90_da.sizes[p90_dim0]}")

    # Remove Feb 29 if needed 
    mask_da = mask_da.convert_calendar('noleap', align_on='year') 
    data_da = data_da.convert_calendar('noleap', align_on='year')

    # Arrays with lat, lon values
    lat, lon = mask_da["lat"].values, mask_da["lon"].values
    n_time, n_lat, n_lon = mask_da.shape

    # Precompute growing season mask by month
    grow_mask_by_month = {m: get_growing_month_mask(lat, lon, m) for m in range(1, 13)}

    years = np.unique(mask_da.time.dt.year.values)
    ny = len(years)

    # These arrays start out empty and end up containing the event-based thirstwave metrics
    sum_event_intensity = np.zeros((ny, n_lat, n_lon), dtype=np.float32)
    event_count = np.zeros((ny, n_lat, n_lon), dtype=np.float32)
    
    sum_event_rad_frac = np.zeros((ny, n_lat, n_lon), dtype=np.float32)
    count_event_rad_frac = np.zeros((ny, n_lat, n_lon), dtype=np.float32)

    true_days = np.zeros((ny, n_lat, n_lon), dtype=np.float32)
    gs_days = np.zeros((ny, n_lat, n_lon), dtype=np.float32)

    for yi, y in enumerate(years):

        yesterday = np.zeros((n_lat, n_lon), dtype=bool) 

        current_event_sum = np.zeros((n_lat, n_lon), dtype=np.float32)
        current_event_count = np.zeros((n_lat, n_lon), dtype=np.float32)

        current_rad_sum = np.zeros((n_lat, n_lon), dtype=np.float32)
        current_rad_count = np.zeros((n_lat, n_lon), dtype=np.float32)

        for m in range(1, 13):

            # Select ET0 data, event days, and 90th percentiles in this month (30 or 31 days)
            data = data_da.sel(time=f"{y}{m:02d}")
            valid_days_month = mask_da.sel(time=f"{y}{m:02d}")
            p90 = p90_da.sel(dayofyear=data.time.dt.dayofyear)
            
            nday = data.shape[0]
            anomaly = data - p90

            # Select fractional contribution of rad term data for this month
            rad_frac = rad_frac_da.sel(time=f"{y}{m:02d}")
            
            # Days this month that are inside a thirstwave event during the growing season 
            valid_days_in_gs = valid_days_month & grow_mask_by_month[m]

            # Sums number of thirswave days in growing season
            true_days[yi, :, :] += valid_days_in_gs.values.sum(axis=0)

            # Sums number of days in growing season 
            # At each grid cell adds 30 or 31 if it is growing season, 0 otherwise
            gs_days[yi, :, :] += (nday * grow_mask_by_month[m].astype(np.float32))

            for di in range(nday):

                today = valid_days_in_gs.isel(time=di).values # is each cell in a thirstwave (and in GS) today?
                anom_today = anomaly.isel(time=di).values # how far above the threshold each cell was today
                
                rad_frac_today = rad_frac.isel(time=di).values # fractional contribution of radiative term in each grid cell today
                finite_today = (today & np.isfinite(rad_frac_today))

                start = today & (~yesterday) # today is TW day and yesterday it wasn't --> Event start
                end = (~today) & yesterday # today is NOT TW day but yesterday it was --> Event end

                valid_end = end & (current_event_count > 0) # positions where events ended

                # Event ended --> add up value of intensity and rad_frac to counters
                sum_event_intensity[yi, valid_end] += (
                    current_event_sum[valid_end]
                    / current_event_count[valid_end]
                )
                event_count[yi, valid_end] += 1

                valid_end_rad = end & (current_rad_count > 0) # positions where events ended

                sum_event_rad_frac[yi, valid_end_rad] += (
                        current_rad_sum[valid_end_rad]
                        / current_rad_count[valid_end_rad]
                )
                count_event_rad_frac[yi, valid_end_rad] += 1

                # Reset current event counters
                current_event_sum[valid_end] = 0
                current_event_count[valid_end] = 0                
                current_rad_sum[valid_end_rad] = 0
                current_rad_count[valid_end_rad] = 0

                # Initialize new events
                current_event_sum[start] = 0
                current_event_count[start] = 0
                current_rad_sum[start] = 0
                current_rad_count[start] = 0

                # Accumulate ongoing events
                current_event_sum[today] += anom_today[today]
                current_event_count[today] += 1
                current_rad_sum[finite_today] += rad_frac_today[finite_today]
                current_rad_count[finite_today] += 1

                yesterday = today

        # finalize events continuing to last day of year
        valid_end = yesterday & (current_event_count > 0)
        sum_event_intensity[yi, valid_end] += (
            current_event_sum[valid_end]
            / current_event_count[valid_end]
        )
        event_count[yi, valid_end] += 1

        valid_end_rad = yesterday & (current_rad_count > 0)
        sum_event_rad_frac[yi, valid_end_rad] += (
                current_rad_sum[valid_end_rad]
                / current_rad_count[valid_end_rad]
        )
        count_event_rad_frac[yi, valid_end_rad] += 1

    # Final event-based metrics
    intensity = sum_event_intensity / event_count
    intensity[event_count == 0] = np.nan

    mean_duration = true_days / event_count
    mean_duration[event_count == 0] = np.nan

    event_freq_100d = event_count / gs_days * 100.0
    event_freq_100d[gs_days == 0] = np.nan

    day_fraction = true_days / gs_days
    day_fraction[gs_days == 0] = np.nan

    rad_frac_tw = np.full((ny, n_lat, n_lon), np.nan, dtype=np.float32)
    valid = count_event_rad_frac > 0
    rad_frac_tw[valid] = sum_event_rad_frac[valid] / count_event_rad_frac[valid]

    # Build output dataset
    coords = ("year", "lat", "lon")
    ds_out = xr.Dataset({
        "intensity": (coords, intensity),
        "event_count": (coords, event_count),
        "mean_duration": (coords, mean_duration),
        "true_days": (coords, true_days),
        "gs_days": (coords, gs_days),
        "event_freq_100d": (coords, event_freq_100d),
        "day_fraction": (coords, day_fraction),
        "rad_frac_tw": (coords, rad_frac_tw),
        },
        coords={"year": years, "lat": lat, "lon": lon}
    )

    # Add metadata
    ds_out["intensity"].attrs["long_name"] = "mean event-based ET0 anomaly during thirstwaves in growing season"
    ds_out["intensity"].attrs["units"] = "mm day-1"

    ds_out["event_count"].attrs["long_name"] = "number of thirstwaves in growing season"
    ds_out["event_count"].attrs["units"] = "events year-1"

    ds_out["mean_duration"].attrs["long_name"] = "mean duration of thirstwave events in growing season"
    ds_out["mean_duration"].attrs["units"] = "days event-1"

    ds_out["true_days"].attrs["long_name"] = "total number of thirstwave days during the growing season"
    ds_out["true_days"].attrs["units"] = "days year-1"

    ds_out["gs_days"].attrs["long_name"] = "length of this year's growing season"
    ds_out["gs_days"].attrs["units"] = "days year-1"

    ds_out["event_freq_100d"].attrs["long_name"] = "thirstwave events per 100 growing-season days"
    ds_out["event_freq_100d"].attrs["units"] = "events per 100 growing-season days"

    ds_out["day_fraction"].attrs["long_name"] = "proportion of growing season days that are thirstwave days"
    ds_out["day_fraction"].attrs["units"] = "%"

    ds_out["rad_frac_tw"].attrs["long_name"] = "mean event-based fractional contribution of radiative term during thirstwaves in growing season"
    ds_out["rad_frac_tw"].attrs["units"] = "1"

    return ds_out
    

In [6]:
rad_frac_ds = xr.open_dataset(filename)
rad_frac_ds

<xarray.Dataset> Size: 16GB
Dimensions:    (time: 60225, lat: 128, lon: 256)
Coordinates:
  * time       (time) object 482kB 1850-01-01 12:00:00 ... 2014-12-31 12:00:00
    height     (time) float64 482kB ...
    dayofyear  (time) int64 482kB ...
  * lat        (lat) float64 1kB -88.93 -87.54 -86.14 ... 86.14 87.54 88.93
  * lon        (lon) float64 2kB 0.0 1.406 2.812 4.219 ... 355.8 357.2 358.6
Data variables:
    rad_frac   (time, lat, lon) float64 16GB ...

In [8]:
ds_out = calculate_event_based_metrics(data_ds, mask_ds, p90_ds, rad_frac_ds)

# Save result to file
out_dir = "/work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/thirstwave_metrics_growing_season/"
out_file = f"{out_dir}/{model}_{exp}_thirstwave_metrics_growing_season2.nc"
ds_out.to_netcdf(out_file)
print(f"✅ Saved metrics to {out_file}")

/tmp/ipykernel_720749/1122969870.py:146: RuntimeWarning: invalid value encountered in divide
  intensity = sum_event_intensity / event_count
/tmp/ipykernel_720749/1122969870.py:149: RuntimeWarning: invalid value encountered in divide
  mean_duration = true_days / event_count


✅ Saved metrics to /work10/archive/CMIP6/CMIP-SSPs/thirstwave_detection/thirstwave_metrics_growing_season//MIROC6_historical_thirstwave_metrics_growing_season2.nc


In [9]:
ds_out

<xarray.Dataset> Size: 173MB
Dimensions:          (year: 165, lat: 128, lon: 256)
Coordinates:
  * year             (year) int64 1kB 1850 1851 1852 1853 ... 2012 2013 2014
  * lat              (lat) float64 1kB -88.93 -87.54 -86.14 ... 87.54 88.93
  * lon              (lon) float64 2kB 0.0 1.406 2.812 ... 355.8 357.2 358.6
Data variables:
    intensity        (year, lat, lon) float32 22MB 0.0004875 ... 0.1468
    event_count      (year, lat, lon) float32 22MB 1.0 1.0 1.0 ... 3.0 3.0 3.0
    mean_duration    (year, lat, lon) float32 22MB 3.0 4.0 4.0 ... 4.0 4.0 4.0
    true_days        (year, lat, lon) float32 22MB 3.0 4.0 4.0 ... 12.0 12.0
    gs_days          (year, lat, lon) float32 22MB 212.0 212.0 ... 214.0 214.0
    event_freq_100d  (year, lat, lon) float32 22MB 0.4717 0.4717 ... 1.402 1.402
    day_fraction     (year, lat, lon) float32 22MB 0.01415 0.01887 ... 0.05607
    rad_frac_tw      (year, lat, lon) float32 22MB 0.7882 0.7836 ... 0.8566